# Goal Recognition as a Deep Learning Task: the GRNet Approach

## Imports 

In [134]:
import numpy as np
from tensorflow.keras.models import load_model
from os.path import join
import pickle
import time
import os
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
import tensorflow.keras.backend as K
from typing import Union
from keras.layers import Layer
from keras import initializers, regularizers, constraints
from keras.initializers import Constant
from keras.losses import BinaryCrossentropy

## Custom Classes

### Network classes

Code from 
*Yang, Z.; Yang, D.; Dyer, C.; He, X.; Smola, A. J.; and Hovy, E. H.* 2016. **Hierarchical Attention Networks for Document Classification**
https://github.com/philipperemy/keras-attention-mechanism

In [135]:
class AttentionWeights(Layer):
    def __init__(self, step_dim,
                 W_regularizer=None, b_regularizer=None,
                 W_constraint=None, b_constraint=None,
                 bias=True, **kwargs):
        self.supports_masking = True
        self.init = initializers.get('glorot_uniform')
        # self.init = initializers.get(Constant(value=1))

        self.W_regularizer = regularizers.get(W_regularizer)
        self.b_regularizer = regularizers.get(b_regularizer)

        self.W_constraint = constraints.get(W_constraint)
        self.b_constraint = constraints.get(b_constraint)

        self.bias = bias
        self.step_dim = step_dim
        self.features_dim = 0
        super(AttentionWeights, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 3

        self.W = self.add_weight(shape=(input_shape[-1],),
                                 initializer=self.init,
                                 name='{}_W'.format(self.name),
                                 regularizer=self.W_regularizer,
                                 constraint=self.W_constraint)
        self.features_dim = input_shape[-1]

        if self.bias:
            self.b = self.add_weight(shape=(input_shape[1],),
                                     initializer='zero',
                                     name='{}_b'.format(self.name),
                                     regularizer=self.b_regularizer,
                                     constraint=self.b_constraint)
        else:
            self.b = None

        self.built = True

    def compute_mask(self, input, input_mask=None):
        return None

    def call(self, x, mask=None):
        features_dim = self.features_dim
        step_dim = self.step_dim

        eij = K.reshape(K.dot(K.reshape(x, (-1, features_dim)),
                        K.reshape(self.W, (features_dim, 1))), (-1, step_dim))

        if self.bias:
            eij += self.b

        eij = K.tanh(eij)

        a = K.exp(eij)

        if mask is not None:
            a *= K.cast(mask, K.floatx())

        a /= K.cast(K.sum(a, axis=1, keepdims=True) + K.epsilon(), K.floatx())

        return a

    def compute_output_shape(self, input_shape):
        return input_shape[0],  self.features_dim

    def get_config(self):
        config={'step_dim':self.step_dim}
        base_config = super(AttentionWeights, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))
    




class ContextVector(Layer):
    def __init__(self, **kwargs):
        super(ContextVector, self).__init__(**kwargs)
        self.features_dim = 0

    def build(self, input_shape):
        assert len(input_shape) == 2
        self.features_dim = input_shape[0][-1]
        self.built = True

    def call(self, x, **kwargs):
        assert len(x) == 2
        h = x[0]
        a = x[1]
        a = K.expand_dims(a)
        weighted_input = h * a
        return K.sum(weighted_input, axis=1)

    def compute_output_shape(self, input_shape):
        return input_shape[0][0], self.features_dim

    def get_config(self):
        base_config = super(ContextVector, self).get_config()
        return dict(list(base_config.items()))

### Constants class

In [136]:
from src.models.loss import *
class C:
    '''
    Constants class.
    '''
    OBSERVATIONS = 0
    CORRECT_GOAL = 1
    POSSIBLE_GOALS = 2 
    
    SATELLITE = 0
    LOGISTICS = 1
    ZENOTRAVEL = 2
    BLOCKSWORLD = 3
    DRIVERLOG = 4
    DEPOTS = 5
    
    MAX_PLAN_LENGTH = 0
    MODEL_FILE = 1
    DICTIONARIES_DICT = 2
    NAME=3
    
    SMALL = 0
    COMPLETE = 1
    PERCENTAGE = 2
    KERAS = 3
    
    #MODELS_DIR = '../../models/models_14/'
    MODELS_DIR = '../../models/models_11/'
    DICTIONARIES_DIR = '../../data/dictionaries'
    #MODELS_DIR = './incremental_models/'
    
    MODEL_LOGISTICS = None
    MODEL_SATELLITE = None
    MODEL_ZENOTRAVEL = None
    MODEL_BLOCKSWORLD = None
    MODEL_DRIVERLOG = None
    MODEL_DEPOTS = None

    MAX_PLAN_PERCENTAGE = 1

    TABLE_HEADERS = ['', 'Pereira', 'Our', 'Support']
    
    CUSTOM_OBJECTS = {'AttentionWeights': AttentionWeights,
                   'ContextVector' : ContextVector,
                   'custom_multilabel_loss_v3' : BinaryCrossentropy,
                   "rmse": rmse,
                "rmse_ol": rmse_ol,
                "rmse_op": rmse_op,
                "rmse_hmlp": rmse_hmlp,
                "rmse_mlp": rmse_mlp,
                "bce_ol": bce_ol,
                "bce_op": bce_op,
                "bce_hmlp": bce_hmlp,
                "bce_mlp": bce_mlp,
                "bfce_ol": bfce_ol,
                "bfce_op": bfce_op,
                "bfce_hmlp": bfce_hmlp,
                "bfce_mlp": bfce_mlp,
                "neg_hamming_metric": neg_hamming_metric,
                }


### Exceptions

In [137]:
class PlanLengthError(Exception):
    pass

class FileFormatError(Exception):
    pass

class UnknownIndexError(Exception):
    pass

## Custom Methods

### Create zip with all the obs

In [138]:
# domains = ['satellite', 'logistics', 'zenotravel', 'blocksworld', 'driverlog', 'depots']

# for domain in domains:
#     os.system(f"mkdir /home/lorenzoserina/OnlineGR/GRNet/testsets/custom_testset/{domain}")
#     results_path=f"/home/lorenzoserina/OnlineGR/GRNet/testsets/custom_testset/{domain}/"
#     path=f"/home/lorenzoserina/OnlineGR/GRNet/testsets/TS_PerGen/{domain}/100/"
#     #For each zip file in this directory, unzip the file, take the obs.dat file and read the lines
#     for filename in os.listdir(path):
#         if filename.endswith(".zip"):
#             os.system(f"mkdir {results_path}{filename[:-4]}")
#             os.system(f"unzip -d {results_path}{filename[:-4]} {path}{filename}")
#             #Read the obs.dat file
#             with open(f"{results_path}{filename[:-4]}/obs.dat", 'r') as f:
#                 lines = f.readlines()
#             for i, line in enumerate(lines):
#                 os.system(f"mkdir {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 #Copy the domain.pddl, template.pddl, hyps.dat, real_hyp.dat in the new directory
#                 os.system(f"cp {results_path}{filename[:-4]}/domain.pddl {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 os.system(f"cp {results_path}{filename[:-4]}/template.pddl {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 os.system(f"cp {results_path}{filename[:-4]}/hyps.dat {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 os.system(f"cp {results_path}{filename[:-4]}/real_hyp.dat {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 #Write a new obs.dat file with only the first i lines
#                 with open(f"{results_path}{filename[:-4]}/{filename[:-4]}_{i+1}/obs.dat", 'w') as f:
#                     for j in lines[:i+1]:
#                         f.write(j)
#                 #Zip the directory
#                 os.system(f"zip -r {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}.zip {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#                 #Remove the directory
#                 # os.system(f"rm -r {results_path}{filename[:-4]}/{filename[:-4]}_{i+1}")
#             #Remove the domain.pddl, template.pddl, hyps.dat, real_hyp.dat
#             os.system(f"rm {results_path}{filename[:-4]}/domain.pddl")
#             os.system(f"rm {results_path}{filename[:-4]}/template.pddl")
#             os.system(f"rm {results_path}{filename[:-4]}/hyps.dat")
#             os.system(f"rm {results_path}{filename[:-4]}/real_hyp.dat")
#             #Remove the obs.dat file
#             os.system(f"rm {results_path}{filename[:-4]}/obs.dat")
            
    

### Unpack files methods

In [139]:
def unzip_file(file_path: str, target_dir: str) -> None:
    '''
    Unzip a file in an empty directory. The directory is 
    emptied before the execution.
    
    Args:
        file_path:
            A string that contains the path
            to the .zip file.
        
        target_dir:
            A string that contains the path 
            to the target directory. This 
            directory is created if it doesn't
            exist and it is emptied if it exists.
        
    '''
    if os.path.exists(target_dir):
        for f in os.listdir(target_dir):
            os.remove(join(target_dir, f))
        os.rmdir(target_dir)
    os.mkdir(target_dir)
    os.system(f'unzip -qq {file_path} -d {target_dir}')
    
def unpack_bz2(file_path: str, target_dir: str) -> None:
    '''
    Unpack a .bz2 file in an empty directory. The directory 
    is emptied before the execution.
    
    Args:
        file_path:
            A string that contains the path
            to the .bz2 file.
        
        target_dir:
            A string that contains the path 
            to the target directory. This 
            directory is created if it doesn't
            exist and it is emptied if it exists.
        
    '''
    if os.path.exists(target_dir):
        for f in os.listdir(target_dir):
            os.remove(join(target_dir, f))
        os.rmdir(target_dir)
    os.mkdir(target_dir)
    os.system(f'tar -xf {file_path} -C {target_dir}')

### Input parse methods

In [140]:
def load_file(file: str, binary: bool = False, use_pickle: bool = False):
    '''
    Get file content from path.
    
    Args:
        file:
            A string that contains the path
            to the file.
        binary:
            Optional. True if the file is a 
            binary file.
        use_pickle:
            Optional. True if the file was 
            saved using pickle.
            
    Returns:
        The content of the file.
    
    Raises:
        FileNotFoundError:
            An error accessing the file
    '''
    operation = 'r'
    if binary:
        operation += 'b'
    with open(file, operation) as rf:
        if use_pickle:
            output = pickle.load(rf)
        else:
            output = rf.readlines()
        rf.close()
    return output
        
        

In [141]:
def parse_file(read_file: str, content_type: int, dictionary: dict = None):
    '''
    Parse different input files.
    
    Args:
        read_file: 
            String containing the path to the file.
        content_type: 
            Integer representing the kind of parse to apply.
                0: observations file,
                1: correct goal file, 
                2: possible goals file
        
    Returns:
        A list of strings that contains the parsed elements.
        
    Raises:
        FileFormatError: 
            An error regarding the action format in 
            the file   
    '''
    
    msg_empty = f'File {read_file} is empty.'
    msg_index = f'Content type {content_type} is unknown.' 
    
    elements = list()
    
    lines = load_file(read_file)
    if len(lines) == 0:
        raise FileFormatError(msg_empty)
    if content_type == C.OBSERVATIONS:
        elements = parse_observations(lines, dictionary)
    elif content_type == C.POSSIBLE_GOALS:
        elements = parse_possible_goals(lines, dictionary)
    elif content_type == C.CORRECT_GOAL:
        elements = parse_correct_goal(lines[0], dictionary)
    else:
        raise UnknownIndexError(msg_index)
    
    if len(elements) > 0:    
        return elements
    else:
        raise FileFormatError(msg_empty)
        

def remove_parentheses(line: str) -> str:
    '''
    Remove parentheses from a string.
    
    Args:
        line: a string that is enclosed in parentheses.
        For example:
        
        "(string example)"
        
    Returns:
        The string without the parenteses.
        None if the string is empty.
        
    Raises:
        FileFormatError: error handling the string
    '''
    
    msg = (f'Error while parsing a line. Expected "(custom '
    +f'text)" but found "{line}"')
    
    line = line.strip()
    if line.startswith('(') and line.endswith(')'):
        element = line[1:-1]
        element = element.strip()
        if len(element) == 0:
            return None
        else:
            return element
    elif len(line) == 0:
        return None
    else:
        raise FileFormatError(msg)
        
def retrieve_from_dict(key: str, dictionary: dict):
    '''
    Return the dictionary value given the key.
    
    Args:
        key:
            A string that is the key.
        dictionary:
            A dict.
            
    Returns:
        The value corresponding to the key.
    
    Raises:
        KeyError:
            An error accessing the dictionary.
    '''
    
    msg_error = f'Key {key.upper()} is not in the dictionary'
    
    try:
        return dictionary[key.upper()]
    except KeyError:
        print(msg_error)
        np.random.seed(47)
        return np.random.randint(0,len(dictionary))

def parse_correct_goal(line: str, goals_dict: dict = None) -> list:
    '''
    Parse the fluents that compose a goal.
    
    Args:
        line: 
            A string that contains one or more 
            fluents in the goal. Fluents are 
            enclosed in parentheses and separated
            by commas. For example:
            
            "(fluent1), (fluent2),  (fluent3)"
        
        goals_dict:
            Optional. A dictionary that maps each 
            fluent to its unique identifier.
    
    Returns:
        A list of strings containing each fluent 
        without parentheses.
        
    Raises:
        FileFormatError:
            An error accessing the file.
    '''
    msg_empty = 'Parsed goal is empty.'
    
    goal = list()
    line = line.strip()
    fluents = line.split(',')
    for f in fluents:
        fluent = remove_parentheses(f)
        if fluent is not None:
            if goals_dict is not None:
                fluent = retrieve_from_dict(fluent, goals_dict)
            goal.append(fluent)
    if len(goal) > 0:
        return goal
    else:
        raise FileFormatError(msg_empty)
    

        
def parse_observations(lines: list, obs_dict: dict = None) -> list:
    '''
    Removes parentheses and empty strings from 
    the observations list.
    
    Args:
        lines: 
            List of strings that contains the 
            observations. Each observation is
            enclosed in parentheses. For 
            example:
            
            ['(observation1)', '', '(observation2)']
        
        obs_dict:
            Optional. A dictionary that maps each 
            observation to its unique identifier.
            
    Returns:
        The input list without parentheses and
        empty strings.
        
    Raises:
        FileFormatError:
            An error accessing the file.
    '''
    msg_empty='Observations list is empty.'
    
    observations = list()
    
    for line in lines:
        observation = remove_parentheses(line)
        if observation is not None:
            if obs_dict is not None:
                observation = retrieve_from_dict(observation, obs_dict)
            observations.append(observation)
    if len(observations)>0:
        return observations
    else:
        raise FileFormatError(msg_empty)

def parse_possible_goals(lines: list, goals_dict: dict = None) -> list:
    '''
    Parse a list of goals.
    
    Args:
        lines:
            A list of strings that contains each
            possible goal.
            
        goals_dict:
            Optional. A dictionary that maps each 
            fluent to its unique identifier.
    
    Returns:
        A list of lists. Each list contains the fluents
        that compose the goal represented as a string.
        
    Raises:
        FileFormatError:
            An error accessing the file.
    '''
    msg_empty='Possible goals list is empty.'
    
    goals=list()
    for line in lines:
        line = line.strip()
        if len(line)>0:
            goals.append(parse_correct_goal(line, goals_dict))
    if len(goals) > 0:
        return goals
    else:
        raise FileFormatError(msg_empty)
            
            

### Model related methods

In [142]:
def parse_domain(domain: Union[str, int]) -> int:
    '''
    Converts domain name into integer
    
    Args:
        domain: 
            A string or an int that represents
            a domain.
    
    Returns:
        An integer associated to a specific domain.
        
    Raises:
        KeyError:
            An error parsing the domain arg.
    '''
    msg = (f'Provided domain {domain} is not supported. '+
           f'Supported domains are: {C.SATELLITE} : satellite, ' +
           f'{C.LOGISTICS} : logistics, {C.BLOCKSWORLD} : blocksworld, ' +
           f'{C.ZENOTRAVEL} : zenotravel, {C.DRIVERLOG}: driverlog,' + 
           f'{C.DEPOTS}: depots.')
           
    if (str(domain).isdigit() and int(domain) == C.SATELLITE) or str(domain).lower().strip() == 'satellite':
        return C.SATELLITE
    elif (str(domain).isdigit() and int(domain) == C.LOGISTICS) or str(domain).lower().strip() == 'logistics':
        return C.LOGISTICS
    elif (str(domain).isdigit() and int(domain) == C.BLOCKSWORLD) or str(domain).lower().strip() == 'blocksworld':
        return C.BLOCKSWORLD
    elif (str(domain).isdigit() and int(domain) == C.ZENOTRAVEL) or str(domain).lower().strip() == 'zenotravel':
        return C.ZENOTRAVEL
    elif (str(domain).isdigit() and int(domain) == C.DRIVERLOG) or str(domain).lower().strip() == 'driverlog':
        return C.DRIVERLOG
    elif (str(domain).isdigit() and int(domain) == C.DEPOTS) or str(domain).lower().strip() == 'depots':
        return C.DEPOTS
    else:
        raise KeyError(msg)

In [143]:
def get_model(domain: int):
    '''
    Loads the model for a specific domain.
    
    Args:
        domain: 
            an integer associated to a specific 
            domain.
            
    Returns:
        The Model loaded for the domain or None
        if there is no model in memory.
        
    Raises:
        KeyError:
            An error parsing the domain arg.
    '''

    msg = (f'Provided domain {domain} is not supported. '+
       f'Supported domains are: {C.SATELLITE} : satellite, ' +
       f'{C.LOGISTICS} : logistics, {C.BLOCKSWORLD} : blocksworld, ' +
       f'{C.ZENOTRAVEL} : zenotravel, {C.DRIVERLOG}: driverlog,' + 
       f'{C.DEPOTS}: depots.')
    
    if domain == C.LOGISTICS:
        return C.MODEL_LOGISTICS
    elif domain == C.SATELLITE:
        return C.MODEL_SATELLITE
    elif domain == C.DEPOTS:
        return C.MODEL_DEPOTS
    elif domain == C.BLOCKSWORLD:
        return C.MODEL_BLOCKSWORLD
    elif domain == C.DRIVERLOG:
        return C.MODEL_DRIVERLOG
    elif domain == C.ZENOTRAVEL:
        return C.MODEL_ZENOTRAVEL
    else:
        raise KeyError(msg)

In [144]:
def get_domain_related(domain: int, element: int, model_type: int = C.SMALL, 
                       percentage: float = 0) -> Union[int, str]:
    '''
    Returns domain related information
    
    Args:
        domain: 
            an integer associated to a specific 
            domain.
        
        element:
            an integer associated to a specific
            piece of information to retrieve.
        
        model_type:
            an integer associated to the type
            of RNN model in use.
        
        percentage:
            a float that represents the model
            percentage to use. Use only with
            model_type = C.PERCENTAGE.
    
    Returns: 
        Max plan size if element=C.MAX_PLAN_LENGTH,
        Model file if element=C.MODEL_FILE
        Dictionaries directory if element=C.DICTIONARIES_DICT
        
    '''
    
    msg = (f'Provided domain {domain} is not supported. '+
           f'Supported domains are: {C.SATELLITE} : satellite, ' +
           f'{C.LOGISTICS} : logistics, {C.BLOCKSWORLD} : blocksworld, ' +
           f'{C.ZENOTRAVEL} : zenotravel.')
    
    
    if domain == C.LOGISTICS:
        v = {
            'max_plan_len' : 50, # 50, 35
            'name' : 'logistics',
        }
    elif domain == C.SATELLITE:
        v = {
            'max_plan_len' : 40,# 40, 28
            'name' : 'satellite',
        }
    elif domain == C.ZENOTRAVEL:
        v = {
            'max_plan_len' : 40,#40, 28
            'name' : 'zenotravel',
        }
    elif domain == C.BLOCKSWORLD:
        v = {
            'max_plan_len' : 75, #75 52
            'name' : 'blocksworld',
        }
    elif domain == C.DRIVERLOG:
        v = {
            'max_plan_len' : 70,#49, 70
            'name' : 'driverlog',
        }
    elif domain == C.DEPOTS:
        v = {
            'max_plan_len' : 50, #50, 44
            'name' : 'depots'
        }
    else:
        raise KeyError(msg)
        
    if element == C.MAX_PLAN_LENGTH:
        return int(v['max_plan_len']*C.MAX_PLAN_PERCENTAGE)
    
    elif element == C.MODEL_FILE:
        if model_type == C.COMPLETE:
            return f'{v["name"]}.h5'
        elif model_type == C.SMALL:
            return f'{v["name"]}_small.h5'
        elif model_type == C.PERCENTAGE:
            return f'{v["name"]}_{int(percentage*100)}perc.h5'
        elif model_type == C.KERAS:
            return f'{v["name"]}.keras'
        
    elif element == C.DICTIONARIES_DICT:
        return join(C.DICTIONARIES_DIR, f'{v["name"]}')
    elif element == C.NAME:
        return v['name']


### Domain component methods

In [145]:
def get_observations_array(observations: list, max_plan_length: int) -> np.ndarray:
    '''
    Create an array of observations index.
    
    Args:
        observations: 
            A list of action names
            
        max_plan_length:
            An integer that contains the maximum size of
            the list that will be considered.
    
    Returns:
        An array that contains the observations' indexes
    '''
    
    WARNING_MSG = (f'The action trace is too long. Only the last {max_plan_length}'+
                 f'actions will be considered.')
    
    observations_array = np.zeros((1, max_plan_length))
    if len(observations) > max_plan_length:
        
        print(WARNING_MSG)
    start_index = max(0, len(observations) - max_plan_length)
    for index, observation in enumerate(observations[start_index:]):
        if index < max_plan_length:
            observations_array[0][index] = int(observation)
    
    return observations_array
        
    
import tensorflow as tf
import psutil
import os
def get_predictions(observations: list, 
                    max_plan_length: int, 
                    domain: int) -> np.ndarray:
    '''
    Return the model predictions.
    
    Args:
        observations:
            A list of action names.
        
        max_plan_length:
            An integer that contains the maximum size of
            the list that will be considered.
        
        domain:
            An integer associated to a specific domain.
    
    Returns:
        The model predictions.
    '''

    # Set up configuration options to use only one CPU
    # config = tf.compat.v1.ConfigProto(
    #     inter_op_parallelism_threads=1,
    #     intra_op_parallelism_threads=1
    # )
    # tf.compat.v1.keras.backend.set_session(tf.compat.v1.Session(config=config))
    
    # Log the RAM usage before prediction
    # process = psutil.Process(os.getpid())
    # init_mem_mb=process.memory_info().rss / 1024 ** 2
    model = get_model(domain)
    
    # model_input = tf.convert_to_tensor(get_observations_array(observations, max_plan_length))
    input_batch = []
    for i in range(len(observations)):
        obs_array = get_observations_array(observations[i], max_plan_length)
        obs_array = np.reshape(obs_array, (max_plan_length,))  # reshape to (35,)
        #print(obs_array)
        input_batch.append(obs_array)
    model_input = tf.convert_to_tensor(input_batch)


    
    # start_time = time.time()
    y_pred = model.predict(model_input)
    # print(y_pred.shape)
    # print(y_pred)
    # tot_time=time.time()-start_time
    

    # Log the RAM usage after prediction
    # fin_mem_mb=process.memory_info().rss / 1024 ** 2
    # print('RAM usage during prediction: {} MB'.format(fin_mem_mb-init_mem_mb))
    # print('Time taken for prediction: {} seconds'.format(tot_time))
    return y_pred

    # model = get_model(domain)
    
    # model_input = tf.convert_to_tensor(get_observations_array(observations, max_plan_length))
    # y_pred = model.predict(model_input)
    # return y_pred



### GR Instance component methods

In [146]:
def get_score(prediction: np.ndarray, possible_goal: list) -> float:
    '''
    Returns the score for a possible goal.
    
    Args:
        prediction:
            An array that contains the model prediction.
        
        possible_goal:
            A list that contains the possible goal indexes.
        
    Returns:
        An float that represents the score of the possible goal.
    '''
    
    score=0
    
    for index in possible_goal:
        score += prediction[int(index)]
    return score

def get_scores(prediction: np.ndarray, possible_goals: list) -> np.ndarray:
    '''
    Returns the scores for all possible goals.
    
    Args:
        prediction:
            An array that contains the model prediction.
        
        possible_goals:
            A list of possible goals; each possible goal is represented as a
            list
        
    Returns:
        An array that contains the score of each of the possible goals.
    '''
    scores = np.zeros((len(possible_goals, )), dtype=float)
    for index, possible_goal in enumerate(possible_goals):
        scores[index] = get_score(prediction, possible_goal)
    return scores
        

def get_max(scores: np.ndarray) -> list:
    '''
    Returns a list with the index (or indexes) of the highest scores.
    
    Args:
        scores:
            An array that contains the scores as floats.
    
    Returns:
        A list thet contains the indexes of the highest score.
    '''
    max_element = -1
    index_max = list()
    for i in range(len(scores)):
        if scores[i] > max_element:
            max_element = scores[i]
            index_max = [i]
        elif scores[i] == max_element:
            index_max.append(i)

    return index_max
    
def get_result(scores: np.ndarray, correct_goal: int) -> bool:
    '''
    Computes if the goal recognition task is successfull.
    
    Args:
        scores:
            An array of floats that contains a score for 
            each possible goal
        correct_goal: 
            An integer that represents the index of the 
            correct goal
            
    Returns:
        True if the maximum score index corresponds to the 
        correct goal index, False otherwise.
    '''
    idx_max_list = get_max(scores)
    if len(idx_max_list) == 1:
        idx_max = idx_max_list[0]
    else:
        print(f'Algorithm chose randomly one of {len(idx_max_list)} equals candidates.')
        idx_max = idx_max_list[np.random.randint(0, len(idx_max_list))]
    if idx_max == correct_goal:
        return True
    else:
        return False
    
def get_correct_goal_idx(correct_goal: list, possible_goals: list) -> int:
    '''
    Conputes the correct goal index.
    
    Args:
        correct_goal:
            A list of strings that contains the correct goal
            fluents.
        possible_goals:
            A list of possible goals; each possible goal is represented as a
            list.
    
    Returns:
        The index of the correct goal in the possible goals list.
        None if the possible goal list does not contain the correct goal.
    '''
    
    for index, possible_goal in enumerate(possible_goals):
        possible_goal = np.sort(possible_goal)
        correct_goal = np.sort(correct_goal)
        if np.array_equal(possible_goal, correct_goal):
            return index
    return None

### GRNet execution methods

In [147]:
def init_models(model_type: int, percentage: float)-> None:
    '''
    Loads in memory all the models.
    
    Args:
        model_type:
            an integer associated to the type
            of RNN model in use.
        
        percentage:
            a float that represents the model
            percentage to use. Use only with
            model_type = C.PERCENTAGE.
    
    Returns:
        None   
    '''
    
    model_file = get_domain_related(C.LOGISTICS, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_LOGISTICS =  load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    
    model_file = get_domain_related(C.SATELLITE, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_SATELLITE = load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    
    model_file = get_domain_related(C.ZENOTRAVEL, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_ZENOTRAVEL = load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    
    model_file = get_domain_related(C.DEPOTS, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_DEPOTS = load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    
    model_file = get_domain_related(C.DRIVERLOG, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_DRIVERLOG =  load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    
    model_file = get_domain_related(C.BLOCKSWORLD, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    C.MODEL_BLOCKSWORLD =  load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)

In [148]:
def run_experiment(obs_file: str, 
            goals_dict_file: Union[str, None],
            actions_dict_file: Union[str, None],
            possible_goals_file: str, 
            correct_goal_file: str, 
            problem_file: str,
            domain: Union[str, int], 
            verbose: int = 0) -> list:
    '''
    Run the goal recognition experiment

    Args:
        obs_file:
            Path of the file that contains the
            observations (plan)

        goals_dict_file:
            Path of the file that contains the
            goals dictionaries. If None it is
            retrieved from its default location.

        actions_dict_file:
            Path of the file that contains the
            actions dictionaries. If None it is
            retrieved from its default location.

        possible_goals_file:
            Path of the file that contains the
            possible goals.

        correct_goal_file:
            Path of the file that contains the
            correct goal.

        domain:
            String that contains the name of the
            domain or integer that corresponds to
            a domain.

        verbose:
            Integer that corresponds to how much
            information is printed. 0 = no info,
            2 = max info

    Returns:
         A list that contains the result, the correct
         goal index and the predicted goal index.
    '''

    domain = parse_domain(domain)
    if goals_dict_file is None:
        goals_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario_goal')
    goals_dict = load_file(goals_dict_file, binary=True, use_pickle=True)
    if actions_dict_file is None:
        actions_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario')
    
    actions_dict = load_file(actions_dict_file, binary=True, use_pickle=True)
    
    # with open(obs_file, 'r') as f:
    #     lines = f.readlines()    
    
        
    observations = parse_file(obs_file, C.OBSERVATIONS, actions_dict)
    #print(observations)
    if verbose > 1:
        print('Observed actions:\n')
        for o in observations:
            print(o)
    
    
    max_plan_length = get_domain_related(domain, C.MAX_PLAN_LENGTH)
    # print(get_domain_related(domain, C.NAME))
    # print(f'Max plan length: {max_plan_length}')
    
    results = list()
    obs_len=len(observations)
    predictions = get_predictions([observations], max_plan_length, domain)
    # init_state=np.zeros(predictions[0][0].shape)
    # with open(problem_file, 'r') as f:
    #     righe=f.read()
    #     stati=righe.split(':init')[1].split('(:goal')[0].replace('\n','').replace('(','').replace(')','').strip().split('\n')
    #     lista_stati=stati[0].split('\t')
    #     for stato in lista_stati:
    #         if stato!='' and stato.upper() in goals_dict.keys():
    #             #Add 1 to init_state in position goals_dict[stato]
    #             init_state[goals_dict[stato.upper()]]=0.2
    
    # actual_pred=init_state
    
    possible_goals = parse_file(possible_goals_file, C.POSSIBLE_GOALS, goals_dict)
    correct_goal = parse_file(correct_goal_file, C.CORRECT_GOAL, goals_dict) 
    correct_goal_idx = get_correct_goal_idx(correct_goal, possible_goals)
    
    ris_passo_passo={}
    ris_mattia={}
    #ris_mattia['init_state']=np.where(init_state==0.2)[0].tolist()
    ris_mattia['correct_goal']=[int(x) for x in correct_goal]
    ris_mattia['possible_goals'] = [[int(x) for x in pos] for pos in possible_goals]
    ris_mattia['predictions']={}
    for i in range(predictions.shape[1]):
        
        ris_mattia['predictions'][str(i)] = predictions[0][i].tolist()
        #If observation[i] exists
        if i+1 > obs_len:
            break
        ris_passo_passo[i]={}
        #actual_pred*=0.9
        actual_pred=predictions[0][i]
        scores = get_scores(actual_pred, possible_goals)
        
            
        if verbose > 1:
            for index, goal in enumerate(possible_goals):
                print(f'{index} - {goal} : {scores[index]}')
        
        
        result = get_result(scores, correct_goal_idx)
        results.append(result)
        ris_passo_passo[i]['correct_goal']=correct_goal_idx
        ris_passo_passo[i]['predicted_goal']=get_max(scores)
        ris_passo_passo[i]['scores']={}
        for s in range(len(scores)):
            ris_passo_passo[i]['scores'][s]=scores[s]
        
        if verbose > 0:
            # print(f'Predicted goal is {get_max(scores)[0]}')
            # print(f'Correct goal is {correct_goal_idx} - {correct_goal}')
            print(f'Observations {observations[i+1]}')
            print(f'Result is {result}')
    # print(f"Length of observations: {len(observations)}")
    # print(f"Length of results: {len(results)}")
        
            
    return [results, correct_goal_idx, get_max(scores)[0], ris_passo_passo, ris_mattia]



In [149]:
def run_experiment_gr(obs_file: str, 
            goals_dict_file: Union[str, None],
            actions_dict_file: Union[str, None],
            possible_goals_file: str, 
            correct_goal_file: str, 
            problem_file: str,
            domain: Union[str, int], 
            verbose: int = 0) -> list:
    '''
    Run the goal recognition experiment

    Args:
        obs_file:
            Path of the file that contains the
            observations (plan)

        goals_dict_file:
            Path of the file that contains the
            goals dictionaries. If None it is
            retrieved from its default location.

        actions_dict_file:
            Path of the file that contains the
            actions dictionaries. If None it is
            retrieved from its default location.

        possible_goals_file:
            Path of the file that contains the
            possible goals.

        correct_goal_file:
            Path of the file that contains the
            correct goal.

        domain:
            String that contains the name of the
            domain or integer that corresponds to
            a domain.

        verbose:
            Integer that corresponds to how much
            information is printed. 0 = no info,
            2 = max info

    Returns:
         A list that contains the result, the correct
         goal index and the predicted goal index.
    '''

    domain = parse_domain(domain)
    if goals_dict_file is None:
        goals_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario_goal')
    goals_dict = load_file(goals_dict_file, binary=True, use_pickle=True)
    if actions_dict_file is None:
        actions_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario')
    
    actions_dict = load_file(actions_dict_file, binary=True, use_pickle=True)
    
    # with open(obs_file, 'r') as f:
    #     lines = f.readlines()    
    
        
    observations = parse_file(obs_file, C.OBSERVATIONS, actions_dict)
    #print(observations)
    if verbose > 1:
        print('Observed actions:\n')
        for o in observations:
            print(o)
    
    
    max_plan_length = get_domain_related(domain, C.MAX_PLAN_LENGTH)
    # print(get_domain_related(domain, C.NAME))
    # print(f'Max plan length: {max_plan_length}')
    
    results = list()
    obs_len=len(observations)
    init_state=np.zeros(len(goals_dict))
    with open(problem_file, 'r') as f:
        righe=f.read().lower()
        #print(righe)
        #stati=righe.split(':init')[1].split('(:goal')[0].replace('\n','').replace('(','').replace(')','').strip().split('\n')
        stati=righe.split(':init')[1].split('(:goal')[0].replace('(','').replace(')','').strip().split('\n')
        #print(stati)
        #lista_stati=stati[0].split('\t')
        lista_stati=stati
        #print(lista_stati)
        for stato in lista_stati:
            if stato!='' and stato.upper() in goals_dict.keys():
                #Add 1 to init_state in position goals_dict[stato]
                init_state[goals_dict[stato.upper()]]=0.2
    possible_goals = parse_file(possible_goals_file, C.POSSIBLE_GOALS, goals_dict)
    correct_goal = parse_file(correct_goal_file, C.CORRECT_GOAL, goals_dict) 
    correct_goal_idx = get_correct_goal_idx(correct_goal, possible_goals)
    
    ris_passo_passo={}
    ris_mattia={}
    ris_mattia['init_state']=np.where(init_state==0.2)[0].tolist()
    ris_mattia['correct_goal']=[int(x) for x in correct_goal]
    ris_mattia['possible_goals'] = [[int(x) for x in pos] for pos in possible_goals]
    ris_mattia['predictions']={}
    batch=[]
    for i in range(obs_len):
        batch.append(observations[:i+1])
        #print(observations[:i+1])
    predictions = get_predictions(batch, max_plan_length, domain)
    print(predictions.shape[0], len(observations))
    #print(predictions[0])
    for i in range(predictions.shape[0]):
        
        chiave=""
        for key, value in actions_dict.items():
            if value == observations[i]:
                chiave=key
        ris_mattia['predictions'][chiave] = predictions[i].tolist()
        #ris_mattia['predictions'][str(i)] = predictions[i].tolist()
        #If observation[i] exists
        
        ris_passo_passo[i]={}
        #actual_pred*=0.9
        scores = get_scores(predictions[i], possible_goals)
        
            
        if verbose > 1:
            for index, goal in enumerate(possible_goals):
                print(f'{index} - {goal} : {scores[index]}')
        
        
        result = get_result(scores, correct_goal_idx)
        results.append(result)
        ris_passo_passo[i]['correct_goal']=correct_goal_idx
        ris_passo_passo[i]['predicted_goal']=get_max(scores)
        ris_passo_passo[i]['scores']={}
        for s in range(len(scores)):
            ris_passo_passo[i]['scores'][s]=scores[s]
        
        if verbose > 0:
            # print(f'Predicted goal is {get_max(scores)[0]}')
            # print(f'Correct goal is {correct_goal_idx} - {correct_goal}')
            print(f'Observations {observations[i+1]}')
            print(f'Result is {result}')
    # print(f"Length of observations: {len(observations)}")
    # print(f"Length of results: {len(results)}")
        
            
    return [results, correct_goal_idx, get_max(scores)[0], ris_passo_passo, ris_mattia]



In [150]:
def run_experiment_reg(obs_file: str, 
            goals_dict_file: Union[str, None],
            actions_dict_file: Union[str, None],
            possible_goals_file: str, 
            correct_goal_file: str, 
            problem_file: str,
            domain: Union[str, int], 
            verbose: int = 0) -> list:
    '''
    Run the goal recognition experiment

    Args:
        obs_file:
            Path of the file that contains the
            observations (plan)

        goals_dict_file:
            Path of the file that contains the
            goals dictionaries. If None it is
            retrieved from its default location.

        actions_dict_file:
            Path of the file that contains the
            actions dictionaries. If None it is
            retrieved from its default location.

        possible_goals_file:
            Path of the file that contains the
            possible goals.

        correct_goal_file:
            Path of the file that contains the
            correct goal.

        domain:
            String that contains the name of the
            domain or integer that corresponds to
            a domain.

        verbose:
            Integer that corresponds to how much
            information is printed. 0 = no info,
            2 = max info

    Returns:
         A list that contains the result, the correct
         goal index and the predicted goal index.
    '''

    domain = parse_domain(domain)
    if goals_dict_file is None:
        goals_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario_goal')
    goals_dict = load_file(goals_dict_file, binary=True, use_pickle=True)
    if actions_dict_file is None:
        actions_dict_file = join(get_domain_related(domain, C.DICTIONARIES_DICT), 'dizionario')
    
    actions_dict = load_file(actions_dict_file, binary=True, use_pickle=True)
    
    # with open(obs_file, 'r') as f:
    #     lines = f.readlines()    
    
        
    observations = parse_file(obs_file, C.OBSERVATIONS, actions_dict)
    # print(observations)
    if verbose > 1:
        print('Observed actions:\n')
        for o in observations:
            print(o)
    
    
    max_plan_length = get_domain_related(domain, C.MAX_PLAN_LENGTH)
    # print(get_domain_related(domain, C.NAME))
    # print(f'Max plan length: {max_plan_length}')
    
    results = list()
    obs_len=len(observations)
    predictions = get_predictions([observations], max_plan_length, domain)
    init_state=np.zeros(predictions[0][0].shape)
    with open(problem_file, 'r') as f:
        righe=f.read().lower()
        #print(righe)
        stati=righe.split(':init')[1].split('(:goal')[0].replace('(','').replace(')','').replace('\t','').strip().split('\n')
        #stati=righe.split(':init')[1].split('(:goal')[0].replace('(','').replace(')','').strip().split('\n')
        #print(stati)
        lista_stati=stati
        #lista_stati=stati
        #print(lista_stati)
        for stato in lista_stati:
            stato=stato.strip()
            if stato!='' and stato.upper() in goals_dict.keys():
                #Add 1 to init_state in position goals_dict[stato]
                init_state[goals_dict[stato.upper()]]=0.2
    
    actual_pred=init_state
    
    possible_goals = parse_file(possible_goals_file, C.POSSIBLE_GOALS, goals_dict)
    correct_goal = parse_file(correct_goal_file, C.CORRECT_GOAL, goals_dict) 
    correct_goal_idx = get_correct_goal_idx(correct_goal, possible_goals)
    
    ris_passo_passo={}
    ris_mattia={}
    ris_mattia['init_state']=np.where(init_state==0.2)[0].tolist()
    ris_mattia['correct_goal']=[int(x) for x in correct_goal]
    ris_mattia['possible_goals'] = [[int(x) for x in pos] for pos in possible_goals]
    ris_mattia['predictions']={}
    
    for i in range(predictions.shape[1]):
        
        # chiave=""
        # for key, value in actions_dict.items():
        #     if value == observations[i]:
        #         chiave=key
        ris_mattia['predictions'][i] = predictions[0][i].tolist()
        #If observation[i] exists
        if i == obs_len-1:
            print(i+1, obs_len)
            break
        ris_passo_passo[i]={}
        #actual_pred*=0.99
        actual_pred+=predictions[0][i]
        scores = get_scores(actual_pred, possible_goals)
        
            
        if verbose > 1:
            for index, goal in enumerate(possible_goals):
                print(f'{index} - {goal} : {scores[index]}')
        
        
        result = get_result(scores, correct_goal_idx)
        results.append(result)
        ris_passo_passo[i]['correct_goal']=correct_goal_idx
        ris_passo_passo[i]['predicted_goal']=get_max(scores)
        ris_passo_passo[i]['scores']={}
        for s in range(len(scores)):
            ris_passo_passo[i]['scores'][s]=scores[s]
        
        if verbose > 0:
            # print(f'Predicted goal is {get_max(scores)[0]}')
            # print(f'Correct goal is {correct_goal_idx} - {correct_goal}')
            print(f'Observations {observations[i+1]}')
            print(f'Result is {result}')
    # print(f"Length of observations: {len(observations)}")
    # print(f"Length of results: {len(results)}")
        
            
    return [results, correct_goal_idx, get_max(scores)[0], ris_passo_passo, ris_mattia]



## GRNet execution

Do not change these values

In [151]:
model_type=C.KERAS#C.SMALL , KERAS
percentage=1

Change these values to fit your execution

In [152]:
domains = [C.BLOCKSWORLD, C.DEPOTS, C.DRIVERLOG, C.LOGISTICS, C.SATELLITE, C.ZENOTRAVEL]
domain = C.BLOCKSWORLD
domain_dir = f'../../GRNet/testsets/custom_dataset/blocksworld/100' #custom_dataset/blocksworld     goal-plan-recognition-dataset/blocks-world
source_dir = f'./files_temp_dir'
verbose = 0

In [153]:
from distutils.dir_util import copy_tree
import numpy as np
import json
from tqdm import tqdm
domains = [C.BLOCKSWORLD, C.DEPOTS, C.DRIVERLOG, C.LOGISTICS, C.SATELLITE, C.ZENOTRAVEL] #C.BLOCKSWORLD, C.DEPOTS, C.DRIVERLOG, C.LOGISTICS, C.SATELLITE, C.ZENOTRAVEL
# with open(f'./results.txt', 'w') as f:
#         f.write(f'Risultati \n\n')
#domains=[C.LOGISTICS]
init_models(model_type, percentage)


for domain in domains:
    ris_path=f"../../results/nicholas/{get_domain_related(domain, C.NAME)}"
    if not os.path.exists(ris_path):
        os.mkdir(ris_path)
    ris_mattia={}
    risultati={}
    times=[]
    if domain == C.BLOCKSWORLD:
        domain_dir = f'../../data/testsets/total_dataset/blocksworld' #custom_dataset/blocksworld     goal-plan-recognition-dataset/blocks-world
    elif domain == C.ZENOTRAVEL:
        domain_dir = f'../../data/testsets/total_dataset/zenotravel' #goal-plan-recognition-dataset/zeno-travel
    else:
        domain_dir = f'../../data/testsets/total_dataset/{get_domain_related(domain, C.NAME)}'
        #domain_dir = f'../../GRNet/testsets/goal-plan-recognition-dataset/{get_domain_related(domain, C.NAME)}'
    # model_file = get_domain_related(C.BLOCKSWORLD, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    # C.MODEL_BLOCKSWORLD = load_model(join(C.MODELS_DIR, model_file), custom_objects=C.CUSTOM_OBJECTS)
    # model_file = get_domain_related(C.ZENOTRAVEL, C.MODEL_FILE, model_type=model_type, percentage=percentage)
    # C.MODEL_ZENOTRAVEL = load_model(join("../../goal_recognition/models/zeno_11_1000/seed_1000", model_file), custom_objects=C.CUSTOM_OBJECTS)
    verbose = 0
    print(f'Running experiment on {get_domain_related(domain, C.NAME)}')
    perc_list = [1] 
    source_dir = './test_instance_pereira/'
    results = list()
    times = list()
    risultati[get_domain_related(domain, C.NAME)]={}
    risultati[get_domain_related(domain, C.NAME)]['RF TOT']=""
    risultati[get_domain_related(domain, C.NAME)]['CV TOT']=""
    risultati[get_domain_related(domain, C.NAME)]['Accuracy']=[]
    RF_list=[]
    CV_list=[]
    accuracy_list=np.zeros((10,))
    plans_dir = f'{join(domain_dir, str(int(1*100)))}'
    files = os.listdir(plans_dir)
    total=0
    correct=0
    results_file = [list(), list()]
    print(f"Numero piani: {len(files)}")
    count_true=0
    count_tot=0
    for j, f in enumerate(files):   
        risultati[get_domain_related(domain, C.NAME)][f]={}  
        #print(f)
        ris_mattia[f]={}
        if f.endswith('.zip'):
            unzip_file(join(plans_dir,f), source_dir)
        elif f.endswith('.bz2'):
            unpack_bz2(join(plans_dir,f), source_dir)
        else:
            #Copy all the files in plans_dir to source_dir
            copy_tree(join(plans_dir,f), source_dir)
        #print(f)
        start_time = time.time()
        result = run_experiment_reg(obs_file=join(source_dir, 'obs.dat'),
                                goals_dict_file=None,
                                actions_dict_file=None,
                                possible_goals_file=join(source_dir, 'hyps.dat'),
                                correct_goal_file=join(source_dir, 'real_hyp.dat'),
                                problem_file=join(source_dir, 'template.pddl'),
                                domain=domain, 
                                verbose=0)
        tot_time=time.time()-start_time
        ris_mattia[f]=result[4]
        
        #print(f"Time: {tot_time}")
        times.append(tot_time/len(result[0]))
        
        RF=0
        CV=0
        #print(result[0])
        #perc=[int(np.ceil(len(result[0])*(i/10))) for i in range(1,11)]
        #print(perc)
        #print(perc)
        acc_perc=[]
        k=1
        for n, i in enumerate(result[0]):
            count_tot+=1
            if i==True:
                count_true+=1
                RF+=1
                CV+=1
            else:
                CV=0
            while n+1 ==int(np.ceil(len(result[0])*(k/10))):
                acc_perc.append(int(i))
                k+=1
        #print(f"Accuracy: {acc_perc}")
        #if acc_perc has just 0s print f
        if sum(acc_perc)==0:
            print(f)
            
        risultati[get_domain_related(domain, C.NAME)][f]['RF']=str(RF/len(result[0]))
        risultati[get_domain_related(domain, C.NAME)][f]['CV']=str(CV/len(result[0]))
        risultati[get_domain_related(domain, C.NAME)][f]['Accuracy']=acc_perc
        risultati[get_domain_related(domain, C.NAME)][f]['StepByStep']=result[3]
        risultati[get_domain_related(domain, C.NAME)][f]['Time']=str(tot_time)
        risultati[get_domain_related(domain, C.NAME)][f]['Correct']=result[1]
        risultati[get_domain_related(domain, C.NAME)][f]['Predicted']=result[2]
        RF_list.append(RF/len(result[0]))
        CV_list.append(CV/len(result[0]))
        accuracy_list=[sum(i) for i in zip(accuracy_list, acc_perc)]
        

    risultati[get_domain_related(domain, C.NAME)]['RF TOT']=str(sum(RF_list)/len(RF_list))
    risultati[get_domain_related(domain, C.NAME)]['CV TOT']=str(sum(CV_list)/len(CV_list))
    risultati[get_domain_related(domain, C.NAME)]['Accuracy']=np.divide(accuracy_list, len(files)).tolist()
    risultati[get_domain_related(domain, C.NAME)]['Times']=str(sum(times)/len(times))
    
    print(f"RF: {str(sum(RF_list)/len(RF_list)).replace('.',',')}")
    print(f'CV: {str(sum(CV_list)/len(CV_list)).replace(".",",")}')
    print(sum(times)/len(times))
    # print(f"Times: {str(sum(times)/len(times)).replace('.',',')}")
    # print(f"Accuracy: {[str(x).replace('.',',') for x in np.divide(accuracy_list, len(files))]}")
    #Write on a json file the results
    # Convert numpy arrays to lists
    
    # for key, value in ris_mattia.items():
    #     if isinstance(value, np.ndarray):
    #         ris_mattia[key] = value.tolist()
    #     elif isinstance(value, dict):
    #         for sub_key, sub_value in value.items():
    #             if isinstance(sub_value, np.ndarray):
    #                 ris_mattia[key][sub_key] = sub_value.tolist()
    # os.system(f"mkdir ../../goal_recognition/results/ogr/{get_domain_related(domain, C.NAME)}")
    with open(f"{ris_path}/clernet.json", 'w') as f:
        json.dump(ris_mattia, f, indent=4)
    

Running experiment on blocksworld
Numero piani: 846
1/1 [==============================] - 0s 458ms/step
34 34
1/1 [==============================] - 0s 98ms/step
28 28
1/1 [==============================] - 0s 74ms/step
42 42
1/1 [==============================] - 0s 54ms/step
48 48
1/1 [==============================] - 0s 56ms/step
62 62
1/1 [==============================] - 0s 50ms/step
52 52
1/1 [==============================] - 0s 49ms/step
18 18
1/1 [==============================] - 0s 50ms/step
20 20
1/1 [==============================] - 0s 50ms/step
8 8
1/1 [==============================] - 0s 50ms/step
32 32
1/1 [==============================] - 0s 48ms/step
8 8
1/1 [==============================] - 0s 49ms/step
14 14
1/1 [==============================] - 0s 55ms/step
12 12
blocksworld_p001716_hyp=hyp-5_100.zip
1/1 [==============================] - 0s 52ms/step
38 38
The action trace is too long. Only the last 75actions will be considered.
1/1 [======================